# COSC726 · Lab 5 — Give Layla Memory
### Real embeddings · a real vector store · no mocks

**Week 6 · ~2.5 hours · self-contained**

Your agent can act. It cannot remember. Today you add four kinds of memory
and then do the harder thing: find out whether any of them helped.

| Part | You build | Kind |
|---|---|---|
| A | Retrieval as a tool | **Task 1** |
| B/C | Episodic and semantic stores | **Task 2** |
| D | Two-user isolation | **Task 3** |
| E | Forgetting: supersession | given |
| F | The ablation | **Task 4** |

### What you are running

**Real `nomic-embed-text` embeddings** from Ollama (768-dim, no API key) and
**real ChromaDB** as the vector store. Nothing here is simulated.

### Two traps this lab makes visible

1. **768 vs 1536.** `nomic-embed-text` returns 768 dimensions; OpenAI's
   returns 1536. Build a collection with one and query with the other and it
   fails — reportedly the commonest reason a local memory setup silently
   breaks. Our retriever refuses the mismatch loudly instead.
2. **Chroma collection names** must be 3–512 characters of `[a-zA-Z0-9._-]`.
   A one-character name raises.

### The rule that matters most

**The two-user isolation test in Part D is not optional.** Cross-user memory
leakage is trivially easy to build — one shared store, no scope on the query
— and it is a data-protection incident rather than a bug.


## Part 0 — Setup

Colab has no Ollama, so this cell installs and starts one, pulls the
embedding model, and writes `lab6_kit.py` into the runtime. Give it three or
four minutes the first time.

In [1]:
# @title Setup — Ollama, the embedding model, and lab6_kit on your PC  { display-mode: "form" }
import os, subprocess, time, urllib.request

!pip -q install chromadb 2>&1 | tail -2

def ollama_up(url="http://localhost:11434"):
    try:
        urllib.request.urlopen(url, timeout=2); return True
    except Exception:
        return False

if not ollama_up():
    print("installing Ollama ...")
    !curl -fsSL https://ollama.com/install.sh | sh 2>&1 | tail -2
    subprocess.Popen(["ollama", "serve"],
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for _ in range(30):
        if ollama_up():
            break
        time.sleep(1)

print("ollama running:", ollama_up())
!ollama pull nomic-embed-text 2>&1 | tail -1

google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.0 which is incompatible.
installing Ollama ...
  - RHEL/CentOS/Fedora: sudo dnf install zstd
  - Arch: sudo pacman -S zstd


FileNotFoundError: [Errno 2] No such file or directory: 'ollama'

In [2]:
# @title Setup — Ollama + ChromaDB + nomic-embed-text on Colab { display-mode: "form" }

import os
import platform
import subprocess
import time
import urllib.request
import urllib.error
import json
import shutil


# ============================================================
# 1. Install Python dependencies
# ============================================================

print("=== 1. Installing Python dependencies ===")

subprocess.run(
    ["pip", "install", "-q", "-U", "chromadb"],
    check=True
)

print("✓ ChromaDB installed")


# ============================================================
# 2. Detect Colab architecture
# ============================================================

print("\n=== 2. Detecting system architecture ===")

machine = platform.machine().lower()

print("Detected architecture:", machine)

if machine in ("x86_64", "amd64"):
    ollama_arch = "amd64"

elif machine in ("aarch64", "arm64"):
    ollama_arch = "arm64"

else:
    raise RuntimeError(
        f"Unsupported architecture: {machine}"
    )

print("✓ Using Ollama architecture:", ollama_arch)


# ============================================================
# 3. Install system dependency: zstd
# ============================================================

print("\n=== 3. Installing zstd ===")

subprocess.run(
    ["apt-get", "update", "-qq"],
    check=True
)

subprocess.run(
    ["apt-get", "install", "-y", "-qq", "zstd"],
    check=True
)

print("✓ zstd installed")


# ============================================================
# 4. Remove broken previous Ollama installation
# ============================================================

print("\n=== 4. Cleaning previous Ollama installation ===")

possible_paths = [
    "/usr/local/bin/ollama",
    "/usr/bin/ollama"
]

for path in possible_paths:
    if os.path.isfile(path):
        try:
            result = subprocess.run(
                [path, "--version"],
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE,
                timeout=5
            )

            if result.returncode != 0:
                print("Removing broken Ollama:", path)
                os.remove(path)

        except (OSError, subprocess.SubprocessError):
            print("Removing invalid Ollama:", path)
            os.remove(path)


# Remove previous Ollama libraries if present
if os.path.isdir("/usr/lib/ollama"):
    print("Removing previous Ollama libraries...")
    shutil.rmtree(
        "/usr/lib/ollama",
        ignore_errors=True
    )


# ============================================================
# 5. Download official Ollama Linux archive
# ============================================================

print("\n=== 5. Downloading Ollama ===")

OLLAMA_DOWNLOAD = (
    f"https://ollama.com/download/"
    f"ollama-linux-{ollama_arch}.tar.zst"
)

ARCHIVE_PATH = f"/tmp/ollama-linux-{ollama_arch}.tar.zst"

print("Download URL:")
print(OLLAMA_DOWNLOAD)

download_result = subprocess.run(
    [
        "curl",
        "-fL",
        "--retry", "3",
        "--retry-delay", "2",
        "-o", ARCHIVE_PATH,
        OLLAMA_DOWNLOAD
    ]
)

if download_result.returncode != 0:
    raise RuntimeError(
        "Failed to download Ollama archive."
    )

if not os.path.exists(ARCHIVE_PATH):
    raise RuntimeError(
        "Ollama archive was not downloaded."
    )

archive_size = os.path.getsize(ARCHIVE_PATH)

print(
    "✓ Downloaded:",
    round(archive_size / (1024**3), 2),
    "GB"
)


# ============================================================
# 6. Extract Ollama into /usr
# ============================================================

print("\n=== 6. Extracting Ollama ===")

extract_result = subprocess.run(
    [
        "tar",
        "--zstd",
        "-xf",
        ARCHIVE_PATH,
        "-C",
        "/usr"
    ]
)

if extract_result.returncode != 0:
    raise RuntimeError(
        "Failed to extract Ollama."
    )

print("✓ Ollama extracted")


# ============================================================
# 7. Find Ollama executable
# ============================================================

ollama_path = shutil.which("ollama")

if ollama_path is None:

    candidates = [
        "/usr/bin/ollama",
        "/usr/local/bin/ollama"
    ]

    for candidate in candidates:
        if os.path.exists(candidate):
            ollama_path = candidate
            break


if ollama_path is None:
    raise RuntimeError(
        "Ollama executable could not be found."
    )


print("\nOllama executable:", ollama_path)


# ============================================================
# 8. Verify Ollama executable
# ============================================================

print("\n=== 7. Verifying Ollama ===")

try:

    version = subprocess.run(
        [ollama_path, "--version"],
        capture_output=True,
        text=True,
        timeout=15
    )

except OSError as e:

    raise RuntimeError(
        f"Ollama executable exists but cannot run: {e}"
    )


print(
    version.stdout.strip()
    or version.stderr.strip()
)


if version.returncode != 0:
    raise RuntimeError(
        "Ollama executable failed verification."
    )


print("✓ Ollama binary works")


# ============================================================
# 9. Helper to check Ollama API
# ============================================================

OLLAMA_URL = "http://127.0.0.1:11434"


def ollama_up():

    try:

        with urllib.request.urlopen(
            OLLAMA_URL,
            timeout=2
        ) as response:

            return response.status == 200

    except Exception:
        return False


# ============================================================
# 10. Start Ollama server
# ============================================================

print("\n=== 8. Starting Ollama server ===")


if not ollama_up():

    log_path = "/tmp/ollama.log"

    log_file = open(
        log_path,
        "w"
    )

    env = os.environ.copy()

    # Important for Colab
    env["OLLAMA_HOST"] = "127.0.0.1:11434"

    ollama_process = subprocess.Popen(
        [ollama_path, "serve"],
        stdout=log_file,
        stderr=subprocess.STDOUT,
        env=env
    )

    print("Waiting for Ollama API...")

    for i in range(60):

        if ollama_up():
            break

        if ollama_process.poll() is not None:

            log_file.close()

            print("\n--- Ollama log ---")

            if os.path.exists(log_path):

                with open(log_path) as f:
                    print(f.read())

            raise RuntimeError(
                "Ollama server stopped unexpectedly."
            )

        time.sleep(1)


if not ollama_up():

    print("\n--- Ollama log ---")

    if os.path.exists("/tmp/ollama.log"):

        with open("/tmp/ollama.log") as f:
            print(f.read())

    raise RuntimeError(
        "Ollama API did not start."
    )


print("✓ Ollama server running")
print("✓ API:", OLLAMA_URL)


# ============================================================
# 11. Pull embedding model
# ============================================================

MODEL_NAME = "nomic-embed-text"

print(
    f"\n=== 9. Pulling {MODEL_NAME} ==="
)


pull_result = subprocess.run(
    [
        ollama_path,
        "pull",
        MODEL_NAME
    ]
)


if pull_result.returncode != 0:

    raise RuntimeError(
        f"Failed to pull {MODEL_NAME}"
    )


print(
    f"✓ {MODEL_NAME} ready"
)


# ============================================================
# 12. Show installed models
# ============================================================

print("\n=== 10. Installed models ===")

subprocess.run(
    [
        ollama_path,
        "list"
    ],
    check=False
)


# ============================================================
# 13. Test embedding API
# ============================================================

print("\n=== 11. Testing embeddings ===")


payload = json.dumps(
    {
        "model": MODEL_NAME,
        "input":
            "Agentic AI systems can use tools, "
            "memory, planning and reasoning."
    }
).encode("utf-8")


request = urllib.request.Request(
    f"{OLLAMA_URL}/api/embed",
    data=payload,
    headers={
        "Content-Type":
            "application/json"
    },
    method="POST"
)


try:

    with urllib.request.urlopen(
        request,
        timeout=180
    ) as response:

        embedding_result = json.loads(
            response
            .read()
            .decode("utf-8")
        )

except Exception as e:

    print(
        "\nEmbedding API failed:",
        e
    )

    print("\n--- Ollama log ---")

    if os.path.exists("/tmp/ollama.log"):

        with open("/tmp/ollama.log") as f:

            lines = f.readlines()

            print(
                "".join(
                    lines[-50:]
                )
            )

    raise


# ============================================================
# 14. Validate embedding
# ============================================================

if "embeddings" not in embedding_result:

    raise RuntimeError(
        "Unexpected embedding response:\n"
        + str(embedding_result)
    )


embedding = (
    embedding_result["embeddings"][0]
)


print("\n✓ Embedding generated successfully")

print(
    "Embedding dimensions:",
    len(embedding)
)

print(
    "First 5 values:",
    embedding[:5]
)


# ============================================================
# 15. Final summary
# ============================================================

print("\n" + "=" * 60)
print("LAB 6 SETUP COMPLETE")
print("=" * 60)

print(
    "Architecture       :",
    ollama_arch
)

print(
    "Ollama executable :",
    ollama_path
)

print(
    "Ollama API        :",
    OLLAMA_URL
)

print(
    "Embedding model   :",
    MODEL_NAME
)

print(
    "Embedding size    :",
    len(embedding)
)

print("=" * 60)

=== 1. Installing Python dependencies ===
✓ ChromaDB installed

=== 2. Detecting system architecture ===
Detected architecture: x86_64
✓ Using Ollama architecture: amd64

=== 3. Installing zstd ===
✓ zstd installed

=== 4. Cleaning previous Ollama installation ===

=== 5. Downloading Ollama ===
Download URL:
https://ollama.com/download/ollama-linux-amd64.tar.zst
✓ Downloaded: 1.33 GB

=== 6. Extracting Ollama ===
✓ Ollama extracted

Ollama executable: /usr/bin/ollama

=== 7. Verifying Ollama ===
✓ Ollama binary works

=== 8. Starting Ollama server ===
Waiting for Ollama API...
✓ Ollama server running
✓ API: http://127.0.0.1:11434

=== 9. Pulling nomic-embed-text ===
✓ nomic-embed-text ready

=== 10. Installed models ===

=== 11. Testing embeddings ===

✓ Embedding generated successfully
Embedding dimensions: 768
First 5 values: [0.0034514759, 0.07980593, -0.13566643, -0.000852613, 0.07672252]

LAB 6 SETUP COMPLETE
Architecture       : amd64
Ollama executable : /usr/bin/ollama
Ollama AP

In [3]:
# @title Write lab6_kit.py into the runtime  { display-mode: "form" }
kit_source = r'''"""
COSC726 Lab 5 — memory and knowledge (support module)
=====================================================
Real embeddings, a real vector store, no mocks and no other lab.

    pip install chromadb openai
    ollama pull qwen2.5:7b
    ollama pull nomic-embed-text        # 768-dim embeddings, no API key
    ollama serve

What this module gives you
--------------------------
    POLICY_DOCS             the corpus to retrieve from
    chunk_fixed / recursive / parent_child      three strategies to compare
    OllamaEmbedder          REAL embeddings via /api/embeddings
    ChromaRetriever         a REAL vector store (in-process, no Docker)
    KeywordRetriever        the lexical baseline you must beat
    EpisodicStore / SemanticStore / ProceduralStore
    MemoryScope             the authorisation key on every read and write
    QUESTIONS               a fixed evaluation set with gold answers
    ablate()                render the with/without table

Self-contained: no import from any other week's lab.

Two traps this module makes visible rather than hiding
------------------------------------------------------
1. `nomic-embed-text` returns 768 dimensions. OpenAI's returns 1536. A
   collection built with one and queried with the other fails -- and this is
   reportedly the commonest reason a local memory setup silently breaks.
   `ChromaRetriever` records its dimension and refuses a mismatch loudly.

2. Chroma collection names must be 3-512 characters of [a-zA-Z0-9._-].
   `"t"` raises. Ours are named for you; yours will not be.
"""
from __future__ import annotations

import json
import math
import os
import re
import urllib.error
import urllib.request
from collections import Counter
from dataclasses import dataclass, field
from typing import Any, Iterable

__all__ = [
    "POLICY_DOCS", "Chunk", "chunk_fixed", "chunk_recursive",
    "chunk_parent_child", "OllamaEmbedder", "ChromaRetriever",
    "KeywordRetriever", "MemoryScope", "ScopeError", "Episode",
    "EpisodicStore", "Fact", "SemanticStore", "Procedure", "ProceduralStore",
    "EvalQuestion", "QUESTIONS", "score_retrieval", "AblationRow", "ablate",
    "OLLAMA_URL", "EMBED_MODEL",
]

OLLAMA_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")
EMBED_MODEL = os.getenv("EMBED_MODEL", "nomic-embed-text")


# ---------------------------------------------------------------------------
# 1. The corpus
# ---------------------------------------------------------------------------

POLICY_DOCS: dict[str, str] = {
    "POL-LATE": (
        "Late delivery policy. An order delivered three or more days after "
        "the promised date qualifies for a ten per cent credit. A credit "
        "changes the customer account and therefore requires approval from a "
        "human supervisor; it may be proposed but never applied directly by "
        "an agent. Orders fewer than three days late do not qualify for any "
        "credit. This policy was certified on 2026-07-01 and supersedes the "
        "version dated 2025-11-14, which used a five-day threshold."),
    "POL-REFUND": (
        "Refund policy. A refund may be issued for goods that arrive damaged "
        "or that were never delivered. Refunds require photographic evidence "
        "for damage claims. A refund is a financial action and always "
        "requires human approval. Refund requests older than ninety days "
        "from the delivery date are declined automatically."),
    "POL-ADDRESS": (
        "Address change policy. A delivery address may be changed while an "
        "order is still at the depot. Once an order is out for delivery the "
        "address cannot be changed and the customer must arrange redelivery "
        "with the courier. Address changes require identity confirmation."),
    "POL-BILLING": (
        "Billing disputes. Duplicate charges, incorrect amounts and "
        "unrecognised transactions are handled by the billing team and not "
        "by support agents. Support agents must escalate billing disputes "
        "immediately without attempting a resolution."),
    "POL-ESCALATE": (
        "Escalation. An agent must escalate to a human when evidence is "
        "insufficient, when a customer has contacted us about the same issue "
        "more than once in sixty days, or when the request falls outside the "
        "agent's remit. Escalation should carry the context gathered so far."),
}


# ---------------------------------------------------------------------------
# 2. Chunking
# ---------------------------------------------------------------------------

@dataclass(frozen=True)
class Chunk:
    doc_id: str
    chunk_id: str
    text: str
    parent_id: str | None = None

    def __repr__(self) -> str:
        return f"<{self.chunk_id}: {self.text[:44]!r}...>"


def _words(text: str) -> list[str]:
    return re.findall(r"[a-z0-9]+", text.lower())


def chunk_fixed(docs: dict[str, str], size: int = 18,
                overlap: int = 0) -> list[Chunk]:
    """Split on a fixed word count. The naive baseline.

    Watch what this does to POL-LATE: the threshold ("three or more days")
    and the consequence ("requires approval") can land in different chunks,
    so neither answers the question alone. That is chunking capping
    everything downstream, and no re-ranker recovers from it.
    """
    out: list[Chunk] = []
    for doc_id, text in docs.items():
        toks = text.split()
        step = max(size - overlap, 1)
        for i in range(0, len(toks), step):
            piece = " ".join(toks[i:i + size])
            if piece:
                out.append(Chunk(doc_id, f"{doc_id}#f{i // step}", piece))
    return out


def chunk_recursive(docs: dict[str, str]) -> list[Chunk]:
    """Split on sentence boundaries. The recommended starting point."""
    out: list[Chunk] = []
    for doc_id, text in docs.items():
        sents = [s.strip() for s in re.split(r"(?<=\.)\s+", text) if s.strip()]
        for i, sent in enumerate(sents):
            out.append(Chunk(doc_id, f"{doc_id}#r{i}", sent))
    return out


def chunk_parent_child(docs: dict[str, str]
                       ) -> tuple[list[Chunk], dict[str, str]]:
    """Embed small children, return the whole parent for context.

    Small for finding, large for understanding -- the dominant production
    pattern, and the resolution of the semantic-chunking paradox.
    """
    children = [Chunk(c.doc_id, c.chunk_id, c.text, parent_id=c.doc_id)
                for c in chunk_recursive(docs)]
    return children, dict(docs)


# ---------------------------------------------------------------------------
# 3. Real embeddings
# ---------------------------------------------------------------------------

class EmbedderError(RuntimeError):
    pass


class OllamaEmbedder:
    """Real embeddings from a local Ollama server. No API key, no network.

    `nomic-embed-text` returns 768 dimensions. If you swap the model you
    change the dimension, and every collection built with the old one is
    invalid. That is not a bug you can patch around -- it is a re-index.
    """

    def __init__(self, model: str = EMBED_MODEL, base_url: str = OLLAMA_URL):
        self.model = model
        self.url = f"{base_url.rstrip('/')}/api/embeddings"
        self._dim: int | None = None
        self._cache: dict[str, list[float]] = {}

    @property
    def dim(self) -> int | None:
        return self._dim

    def embed(self, text: str) -> list[float]:
        if text in self._cache:
            return self._cache[text]
        payload = json.dumps({"model": self.model, "prompt": text}).encode()
        req = urllib.request.Request(
            self.url, data=payload,
            headers={"Content-Type": "application/json"})
        try:
            with urllib.request.urlopen(req, timeout=60) as resp:
                vec = json.loads(resp.read())["embedding"]
        except urllib.error.URLError as exc:
            raise EmbedderError(
                f"cannot reach Ollama at {self.url}. Is `ollama serve` "
                f"running, and have you run `ollama pull {self.model}`? "
                f"({exc})") from None
        if self._dim is None:
            self._dim = len(vec)
        elif len(vec) != self._dim:
            raise EmbedderError(
                f"dimension changed mid-run: {len(vec)} vs {self._dim}")
        self._cache[text] = vec
        return vec

    def embed_many(self, texts: Iterable[str]) -> list[list[float]]:
        return [self.embed(t) for t in texts]


# ---------------------------------------------------------------------------
# 4. Retrieval
# ---------------------------------------------------------------------------

class KeywordRetriever:
    """Lexical retrieval with IDF weighting. The baseline you must beat.

    On a corpus this small it is genuinely hard to beat -- which is the
    point. Measure before you reach for embeddings.
    """

    name = "keyword"

    def __init__(self, chunks: Iterable[Chunk],
                 parents: dict[str, str] | None = None):
        self.chunks = list(chunks)
        self.parents = parents
        self.tokens = [Counter(_words(c.text)) for c in self.chunks]
        df: Counter[str] = Counter()
        for t in self.tokens:
            df.update(set(t))
        n = max(len(self.chunks), 1)
        self.idf = {w: math.log(1 + n / (1 + c)) for w, c in df.items()}

    def retrieve(self, query: str, k: int = 3) -> list[Chunk]:
        q = _words(query)
        scored = []
        for chunk, toks in zip(self.chunks, self.tokens):
            score = sum(toks.get(w, 0) * self.idf.get(w, 0.0) for w in q)
            score /= math.sqrt(sum(toks.values()) or 1)
            scored.append((score, chunk))
        scored.sort(key=lambda p: (-p[0], p[1].chunk_id))
        return [c for s, c in scored[:k] if s > 0]


class ChromaRetriever:
    """A real vector store: ChromaDB, in-process, no Docker.

    Note the collection name rule: 3-512 characters of [a-zA-Z0-9._-],
    starting and ending alphanumeric. A one-character name raises.
    """

    name = "chroma"

    def __init__(self, chunks: Iterable[Chunk], embedder: OllamaEmbedder,
                 parents: dict[str, str] | None = None,
                 collection: str = "cosc726-policies"):
        import chromadb

        self.chunks = list(chunks)
        self.parents = parents
        self.embedder = embedder
        self.client = chromadb.EphemeralClient()
        try:
            self.client.delete_collection(collection)
        except Exception:
            pass
        # cosine, not the L2 default: we care about direction, not magnitude
        self.col = self.client.get_or_create_collection(
            collection, metadata={"hnsw:space": "cosine"})

        vecs = embedder.embed_many(c.text for c in self.chunks)
        self.dim = len(vecs[0]) if vecs else 0
        self.col.add(
            ids=[c.chunk_id for c in self.chunks],
            embeddings=vecs,
            documents=[c.text for c in self.chunks],
            metadatas=[{"doc_id": c.doc_id,
                        "parent_id": c.parent_id or ""} for c in self.chunks])

    def retrieve(self, query: str, k: int = 3) -> list[Chunk]:
        qv = self.embedder.embed(query)
        if len(qv) != self.dim:
            raise EmbedderError(
                f"query embedding is {len(qv)}-dim but the collection is "
                f"{self.dim}-dim. You changed embedding model; re-index.")
        res = self.col.query(query_embeddings=[qv], n_results=k)
        by_id = {c.chunk_id: c for c in self.chunks}
        return [by_id[i] for i in res["ids"][0] if i in by_id]


# ---------------------------------------------------------------------------
# 5. Memory stores, scoped
# ---------------------------------------------------------------------------

@dataclass(frozen=True)
class MemoryScope:
    """The authorisation key carried on EVERY read and write.

    Memory access is an authorisation decision, not a lookup. A store that
    accepts a query without a scope is a cross-user leak waiting to happen.
    """
    user_id: str | None = None
    agent_id: str = "layla"
    session_id: str | None = None

    def matches(self, other: "MemoryScope") -> bool:
        if self.user_id is not None and self.user_id != other.user_id:
            return False
        return self.agent_id == other.agent_id


class ScopeError(PermissionError):
    """Raised on a scopeless access. Deliberately loud."""


def _require(scope: MemoryScope | None) -> MemoryScope:
    if scope is None or scope.user_id is None:
        raise ScopeError(
            "every memory read and write must carry a user scope; a "
            "scopeless query is how customer A's data reaches customer B")
    return scope


@dataclass
class Episode:
    episode_id: str
    scope: MemoryScope
    when: str
    what: str
    action: str
    outcome: str
    provenance: list[str] = field(default_factory=list)


class EpisodicStore:
    """What happened, when, and with what outcome."""

    def __init__(self) -> None:
        self._items: list[Episode] = []
        self._n = 2290

    def write(self, scope: MemoryScope, what: str, action: str, outcome: str,
              when: str, provenance: list[str] | None = None) -> Episode:
        _require(scope)
        self._n += 1
        ep = Episode(f"ep-{self._n}", scope, when, what, action, outcome,
                     provenance or [])
        self._items.append(ep)
        return ep

    def recall(self, scope: MemoryScope, about: str = "",
               k: int = 5) -> list[Episode]:
        _require(scope)
        hits = [e for e in self._items if e.scope.matches(scope)]
        if about:
            terms = set(_words(about))
            hits = [e for e in hits
                    if terms & set(_words(e.what + " " + e.action))]
        return sorted(hits, key=lambda e: e.when, reverse=True)[:k]

    def count_since(self, scope: MemoryScope, about: str, since: str) -> int:
        """How many times lately? The 'again' question, made checkable."""
        return len([e for e in self.recall(scope, about, k=99)
                    if e.when >= since])


@dataclass
class Fact:
    key: str
    value: str
    scope: MemoryScope
    asserted: str
    superseded_by: str | None = None

    @property
    def active(self) -> bool:
        return self.superseded_by is None


class SemanticStore:
    """Facts, with supersession rather than duplication."""

    def __init__(self) -> None:
        self._items: list[Fact] = []

    def write(self, scope: MemoryScope, key: str, value: str,
              asserted: str) -> Fact:
        _require(scope)
        for f in self._items:
            if f.key == key and f.scope.matches(scope) and f.active:
                f.superseded_by = asserted          # supersede, not duplicate
        fact = Fact(key, value, scope, asserted)
        self._items.append(fact)
        return fact

    def get(self, scope: MemoryScope, key: str) -> Fact | None:
        _require(scope)
        hits = [f for f in self._items
                if f.key == key and f.scope.matches(scope) and f.active]
        return hits[-1] if hits else None

    def history(self, key: str) -> list[Fact]:
        return [f for f in self._items if f.key == key]


@dataclass
class Procedure:
    rule_id: str
    when: str
    then: str
    status: str = "candidate"            # candidate | active
    provenance: list[str] = field(default_factory=list)


class ProceduralStore:
    """Rules, with a status field so a candidate cannot masquerade as law."""

    def __init__(self) -> None:
        self._items: dict[str, Procedure] = {}

    def propose(self, rule_id: str, when: str, then: str,
                provenance: list[str]) -> Procedure:
        p = Procedure(rule_id, when, then, "candidate", provenance)
        self._items[rule_id] = p
        return p

    def review(self, rule_id: str, approve: bool) -> Procedure | None:
        p = self._items.get(rule_id)
        if p is None:
            return None
        if not approve:
            self._items.pop(rule_id)
            return None
        p.status = "active"
        return p

    def active(self) -> list[Procedure]:
        return [p for p in self._items.values() if p.status == "active"]


# ---------------------------------------------------------------------------
# 6. Evaluation
# ---------------------------------------------------------------------------

@dataclass(frozen=True)
class EvalQuestion:
    qid: str
    question: str
    gold_docs: set[str]
    gold_answer_contains: list[str]
    note: str = ""


QUESTIONS: list[EvalQuestion] = [
    EvalQuestion("Q1", "How many days late does an order have to be to "
                       "qualify for a credit?",
                 {"POL-LATE"}, ["three", "3"],
                 "Single-fact lookup. Everything should get this."),
    EvalQuestion("Q2", "Can an agent apply a credit itself?",
                 {"POL-LATE"}, ["approval", "human"],
                 "The threshold and the authority are different sentences. "
                 "Fixed-size chunking splits them."),
    EvalQuestion("Q3", "A customer says their parcel arrived smashed. What "
                       "do they need to provide?",
                 {"POL-REFUND"}, ["photograph", "evidence"],
                 "Paraphrase: 'smashed' never appears in the corpus."),
    EvalQuestion("Q4", "The order is already out for delivery. Can we change "
                       "the address?",
                 {"POL-ADDRESS"}, ["cannot", "redelivery", "courier"],
                 "Requires the negative case, not just the topic."),
    EvalQuestion("Q5", "Customer reports being charged twice. What now?",
                 {"POL-BILLING"}, ["escalate", "billing team"],
                 "'Charged twice' vs 'duplicate charges' -- no shared word. "
                 "This is where lexical retrieval fails."),
    EvalQuestion("Q6", "This is the customer's second complaint this month "
                       "about the same order. Does that change anything?",
                 {"POL-ESCALATE"}, ["escalate", "more than once"],
                 "Needs BOTH retrieval and episodic memory."),
]


def score_retrieval(retriever: Any, questions: Iterable[EvalQuestion],
                    k: int = 3) -> dict[str, float]:
    """Retrieval recall and precision, measured SEPARATELY from answers.

    Report only end-to-end quality and you cannot tell which stage failed.
    """
    qs = list(questions)
    hits, prec = 0, 0.0
    for q in qs:
        got = retriever.retrieve(q.question, k=k)
        docs = {c.doc_id for c in got}
        if q.gold_docs & docs:
            hits += 1
        prec += (len(q.gold_docs & docs) / len(docs)) if docs else 0.0
    n = max(len(qs), 1)
    return {"recall": hits / n, "precision": prec / n, "k": k}


@dataclass
class AblationRow:
    name: str
    recall: float
    precision: float
    answered: float
    tokens: int

    def as_row(self) -> str:
        return (f"{self.name:<28}{self.recall:>8.0%}{self.precision:>11.0%}"
                f"{self.answered:>10.0%}{self.tokens:>10}")


def ablate(rows: list[AblationRow]) -> str:
    hdr = (f"{'configuration':<28}{'recall':>8}{'precision':>11}"
           f"{'answered':>10}{'tokens':>10}")
    out = [hdr, "-" * len(hdr)] + [r.as_row() for r in rows]
    out.append("")
    out.append("Expect at least one layer to make things worse. That is a "
               "finding,\nnot a bug -- report it.")
    return "\n".join(out)
'''

with open("lab6_kit.py", "w", encoding="utf-8") as f:
    f.write(kit_source)

import importlib, sys
sys.modules.pop("lab6_kit", None)
import lab6_kit as K
importlib.reload(K)

print(f"lab6_kit.py written: {len(kit_source.splitlines())} lines")
print("corpus:", list(K.POLICY_DOCS))
print("questions:", len(K.QUESTIONS))

lab6_kit.py written: 521 lines
corpus: ['POL-LATE', 'POL-REFUND', 'POL-ADDRESS', 'POL-BILLING', 'POL-ESCALATE']
questions: 6


### Prove the embeddings actually work

One call, before anything depends on it. If this fails, nothing below will
work and the error will be less clear.

In [4]:
import json
from lab6_kit import (MemoryScope, EpisodicStore, SemanticStore,
                      ProceduralStore, ScopeError, AblationRow)

embedder = K.OllamaEmbedder()
v = embedder.embed("late delivery policy")
print(f"model: {embedder.model}")
print(f"dimensions: {len(v)}")
print(f"first five: {[round(x, 4) for x in v[:5]]}")
print("\nRemember this number. Change the embedding model and every")
print("collection you have built becomes invalid.")

model: nomic-embed-text
dimensions: 768
first five: [-0.477, 0.7363, -3.9713, 0.1515, -0.2066]

Remember this number. Change the embedding model and every
collection you have built becomes invalid.



## Part 1 — Chunking, before anything else

The lecture claimed chunking caps everything downstream. Test it.

In [5]:
fixed = K.chunk_fixed(K.POLICY_DOCS)
recursive = K.chunk_recursive(K.POLICY_DOCS)
children, parents = K.chunk_parent_child(K.POLICY_DOCS)

print(f"fixed {len(fixed)}  recursive {len(recursive)}  parent-child {len(children)}\n")

# What did fixed-size chunking do to the late-delivery policy?
for ch in fixed:
    if ch.doc_id == "POL-LATE":
        print(ch.chunk_id, "|", ch.text)

fixed 16  recursive 20  parent-child 20

POL-LATE#f0 | Late delivery policy. An order delivered three or more days after the promised date qualifies for a ten
POL-LATE#f1 | per cent credit. A credit changes the customer account and therefore requires approval from a human supervisor; it
POL-LATE#f2 | may be proposed but never applied directly by an agent. Orders fewer than three days late do not
POL-LATE#f3 | qualify for any credit. This policy was certified on 2026-07-01 and supersedes the version dated 2025-11-14, which used
POL-LATE#f4 | a five-day threshold.


**Look at those chunks.** The three-day threshold and the requirement for
human approval are two different facts in POL-LATE. Did they survive in the
same chunk?

That is question **Q2** — *"can an agent apply a credit itself?"* — and it is
the diagnostic for this whole stage.

In [6]:
for label, chunks in [("fixed", fixed), ("recursive", recursive)]:
    kw = K.KeywordRetriever(chunks)
    ch = K.ChromaRetriever(chunks, embedder, collection=f"cosc726-{label}")
    for r in (kw, ch):
        s = K.score_retrieval(r, K.QUESTIONS)
        print(f"{label:<10} {r.name:<9} recall {s['recall']:>4.0%}"
              f"   precision {s['precision']:>4.0%}")

fixed      keyword   recall  67%   precision  33%
fixed      chroma    recall 100%   precision  61%
recursive  keyword   recall  83%   precision  47%
recursive  chroma    recall 100%   precision  64%


### The other diagnostic — where embeddings earn their cost

**Q5** asks about being *"charged twice"*. The policy says *"duplicate
charges"*. No word is shared.

In [7]:
q5 = [q for q in K.QUESTIONS if q.qid == "Q5"][0]
print(q5.question, "\n")
for r in (K.KeywordRetriever(recursive),
          K.ChromaRetriever(recursive, embedder, collection="cosc726-q5")):
    got = [c.doc_id for c in r.retrieve(q5.question, k=3)]
    mark = "HIT " if set(got) & q5.gold_docs else "MISS"
    print(f"  {r.name:<9} {mark} {got}")

# Q. One retriever misses entirely — and returns three plausible documents
#    while doing so. Which failure would you rather debug in production:
#    an empty result, or a confident wrong one?

Customer reports being charged twice. What now? 

  keyword   MISS ['POL-ADDRESS', 'POL-LATE', 'POL-ESCALATE']
  chroma    HIT  ['POL-BILLING', 'POL-BILLING', 'POL-BILLING']


### Task 1 — retrieval as a tool

A Week 4 tool: read tier, structured errors, short results.

> **TODO:** implement `make_search_tool`.

In [8]:
def make_search_tool(retriever, parents=None):
    """Create a scoped search result from retrieved policy passages."""

    def search_policy(query, k=3):
        if not isinstance(query, str) or len(query.strip()) < 3:
            return {"ok": False, "error": "Query is too short."}

        if not isinstance(k, int) or k < 1:
            return {"ok": False, "error": "k must be a positive integer."}

        chunks = retriever.retrieve(query.strip(), k=k)

        if not chunks:
            return {"ok": False, "error": "No matching policy found."}

        passages = []
        for chunk in chunks:
            passages.append({
                "doc_id": chunk.doc_id,
                "chunk_id": chunk.chunk_id,
                "text": (
                    parents[chunk.parent_id]
                    if parents is not None and chunk.parent_id in parents
                    else chunk.text
                ),
            })

        return {
            "ok": True,
            "passages": passages,
            "evidence_ids": [chunk.chunk_id for chunk in chunks],
        }

    return search_policy


tool = make_search_tool(
    K.ChromaRetriever(
        children,
        embedder,
        parents=parents,
        collection="cosc726-pc"
    ),
    parents
)

print(json.dumps(
    tool("how many days late qualifies for a credit"),
    indent=1
)[:500])

{
 "ok": true,
 "passages": [
  {
   "doc_id": "POL-LATE",
   "chunk_id": "POL-LATE#r3",
   "text": "Late delivery policy. An order delivered three or more days after the promised date qualifies for a ten per cent credit. A credit changes the customer account and therefore requires approval from a human supervisor; it may be proposed but never applied directly by an agent. Orders fewer than three days late do not qualify for any credit. This policy was certified on 2026-07-01 and supersedes the 


In [ ]:
# @title ✅ Solution — Task 1  { display-mode: "form" }


{
 "ok": true,
 "passages": [
  {
   "doc_id": "POL-LATE",
   "chunk_id": "POL-LATE#r3",
   "text": "Late delivery policy. An order delivered three or more days after the promised date qualifies for a ten per cent credit. A credit changes the customer account and therefore requires approval from a human supervisor; it may be proposed but never applied directly by an agent. Orders fewer than three days late do not qualify for any credit. This policy was certified on 2026-07-01 and supersedes the 



## Part 2 — Task 2: the stores

Three stores, three jobs. Note what each write requires.

> **TODO:** implement `seed_memory`.

In [9]:
def seed_memory():
    ep = EpisodicStore()
    sem = SemanticStore()
    proc = ProceduralStore()

    layla = MemoryScope(user_id="cust-4417")
    other = MemoryScope(user_id="cust-9001")

    # Episodic memory: what happened to each customer?
    first = ep.write(
        layla,
        what="late delivery of order 4417",
        action="customer contacted support",
        outcome="case escalated for review",
        when="2026-07-10",
        provenance=["support-case-101"],
    )
    second = ep.write(
        layla,
        what="late delivery of order 4417",
        action="customer contacted support again",
        outcome="case escalated for review",
        when="2026-07-25",
        provenance=["support-case-102"],
    )
    ep.write(
        other,
        what="address change for order 9001",
        action="customer contacted support",
        outcome="address updated at depot",
        when="2026-07-20",
        provenance=["support-case-201"],
    )

    # Semantic memory: the newer preference supersedes the older one.
    sem.write(
        layla, "contact_preference", "email",
        asserted="2026-07-01"
    )
    sem.write(
        layla, "contact_preference", "phone",
        asserted="2026-07-26"
    )
    sem.write(
        other, "contact_preference", "email",
        asserted="2026-07-20"
    )

    # Procedural memory: propose a rule with traceable evidence,
    # then record its review.
    proc.propose(
        rule_id="repeat-late-delivery",
        when="the same customer reports late delivery more than once in 60 days",
        then="escalate to a human with the context from both contacts",
        provenance=[first.episode_id, second.episode_id],
    )
    proc.review("repeat-late-delivery", approve=True)

    return ep, sem, proc


ep, sem, proc = seed_memory()
scope = MemoryScope(user_id="cust-4417")
print("episodes for cust-4417:", len(ep.recall(scope, k=99)))

episodes for cust-4417: 2


In [ ]:
# @title ✅ Solution — Task 2  { display-mode: "form" }


episodes for cust-4417: 2



## Part 3 — Task 3: two-user isolation

**The non-negotiable part.** Prove Alice's memory never surfaces for Bob.

> **TODO:** write the assertions.

In [10]:
def isolation_test(ep, sem):
    alice = MemoryScope(user_id="cust-4417")
    bob = MemoryScope(user_id="cust-9001")

    alice_episodes = ep.recall(alice, k=99)
    bob_episodes = ep.recall(bob, k=99)

    assert alice_episodes and bob_episodes
    assert all(e.scope.user_id == alice.user_id for e in alice_episodes)
    assert all(e.scope.user_id == bob.user_id for e in bob_episodes)

    alice_ids = {e.episode_id for e in alice_episodes}
    bob_ids = {e.episode_id for e in bob_episodes}
    assert alice_ids.isdisjoint(bob_ids)

    assert sem.get(alice, "contact_preference").value == "phone"
    assert sem.get(bob, "contact_preference").value == "email"

    try:
        ep.recall(None)
    except ScopeError:
        pass
    else:
        raise AssertionError("A scopeless episode read was allowed")

    try:
        sem.get(None, "contact_preference")
    except ScopeError:
        pass
    else:
        raise AssertionError("A scopeless fact read was allowed")


isolation_test(ep, sem)
print("isolation holds")

isolation holds


In [ ]:
# @title ✅ Solution — Task 3  { display-mode: "form" }


  Alice sees 2, Bob sees 1, no overlap  ✓
  semantic facts scoped independently  ✓
  scopeless read refused  ✓
isolation holds


**What you proved, and what you did not:**

1. Reads are scoped. Are *writes*? What happens if a write carries the wrong
   `user_id`?
2. `ScopeError` fires on a scopeless read. Why is raising better than quietly
   returning an empty list?
3. Where should this check live in a real system — in the store, in the tool,
   or in the agent loop? What does each choice cost?


## Part 4 — Forgetting

Supersession, not duplication. An agent that appends every new fact ends up
holding the old preference and the new one, with nothing marking which is
current.

In [12]:
for f in sem.history("contact_preference"):
    if f.scope.user_id == "cust-4417":
        state = "ACTIVE" if f.active else f"superseded {f.superseded_by}"
        print(f"{f.asserted}  {f.value:<6} {state}")

print("\ncurrent:", sem.get(scope, "contact_preference").value)

# Q. Write out what the agent would say if BOTH facts were active and it
#    retrieved them in the wrong order. That sentence is the bug.

2026-07-01  email  superseded 2026-07-26
2026-07-26  phone  ACTIVE

current: phone


### Consolidation

An episode becomes a candidate rule; a human reviews it; only then is it
active. The `status` field is what stops a one-off workaround becoming
standing policy the agent then cites forever.

In [14]:
for p in proc.active():
    print(f"{p.rule_id}  [{p.status}]")
    print(f"  when {p.when}")
    print(f"  then {p.then}")
    print(f"  justified by {p.provenance}")

n = ep.count_since(scope, "late delivery", "2026-05-29")
print(f"\nlate deliveries since 2026-05-29: {n}  ->  rule fires: {n >= 2}")

# Q. The rule cites two episodes. Ask what a reviewer would ask: do two
#    episodes justify a standing rule, or have you promoted a coincidence?

repeat-late-delivery  [active]
  when the same customer reports late delivery more than once in 60 days
  then escalate to a human with the context from both contacts
  justified by ['ep-2291', 'ep-2292']

late deliveries since 2026-05-29: 2  ->  rule fires: True



## Part 5 — Task 4: the ablation

The assessed part. Run with and without each layer, on the same questions.

**Predict the table before you run it.** Write down which layer will help
most, which will make things worse, and what the token cost will be.

> **TODO:** implement `answer` and `run_ablation`.

In [15]:
def answer(q, retriever, parents, ep, sem, proc, scope,
           use_episodic, use_semantic, use_procedural):
    """Return the evidence context and an approximate token count."""
    if retriever is None:
        return "insufficient evidence", 0

    chunks = retriever.retrieve(q.question, k=3)
    found_authority = bool({c.doc_id for c in chunks} & q.gold_docs)

    lines = []
    for chunk in chunks:
        passage = (
            parents[chunk.parent_id]
            if parents and chunk.parent_id in parents
            else chunk.text
        )
        lines.append(f"[POLICY {chunk.doc_id}/{chunk.chunk_id}] {passage}")

    repeat_confirmed = False
    if use_episodic:
        episodes = ep.recall(scope, k=5)
        for item in episodes:
            lines.append(
                f"[EPISODE {item.episode_id}] "
                f"{item.when}: {item.what}; {item.action}; {item.outcome}"
            )
        repeat_confirmed = (
            ep.count_since(scope, "late delivery", "2026-05-29") >= 2
        )

    if use_semantic:
        fact = sem.get(scope, "contact_preference")
        if fact:
            lines.append(f"[FACT] contact_preference: {fact.value}")

    if use_procedural:
        for rule in proc.active():
            lines.append(
                f"[PROCEDURE {rule.rule_id}] "
                f"when {rule.when}; then {rule.then}"
            )

    context = "\n".join(lines)
    approx_tokens = round(len(context.split()) * 1.33)

    has_answer_term = any(
        term.lower() in context.lower()
        for term in q.gold_answer_contains
    )

    # Q6 needs evidence of this customer's repeated contacts,
    # in addition to the authoritative escalation policy.
    can_answer = found_authority and has_answer_term
    if q.qid == "Q6":
        can_answer = can_answer and repeat_confirmed

    return (
        context if can_answer else "insufficient evidence",
        approx_tokens,
    )


def run_ablation(embedder):
    fixed = K.chunk_fixed(K.POLICY_DOCS)
    recursive = K.chunk_recursive(K.POLICY_DOCS)
    children, parents = K.chunk_parent_child(K.POLICY_DOCS)

    configurations = [
        ("no memory at all", None, None, False, False, False),
        ("fixed + keyword",
         K.KeywordRetriever(fixed), None, False, False, False),
        ("recursive + keyword",
         K.KeywordRetriever(recursive), None, False, False, False),
        ("recursive + embeddings",
         K.ChromaRetriever(
             recursive, embedder, collection="ablation-recursive"
         ), None, False, False, False),
    ]

    pc_retriever = K.ChromaRetriever(
        children, embedder, parents=parents,
        collection="ablation-parent-child"
    )

    configurations.extend([
        ("parent-child + embeddings",
         pc_retriever, parents, False, False, False),
        ("+ episodic",
         pc_retriever, parents, True, False, False),
        ("+ episodic + semantic",
         pc_retriever, parents, True, True, False),
        ("everything",
         pc_retriever, parents, True, True, True),
    ])

    rows = []
    for name, retriever, parent_docs, use_ep, use_sem, use_proc in configurations:
        if retriever is None:
            recall = precision = 0.0
        else:
            scores = K.score_retrieval(retriever, K.QUESTIONS)
            recall, precision = scores["recall"], scores["precision"]

        correct = 0
        total_tokens = 0

        for q in K.QUESTIONS:
            result, tokens = answer(
                q, retriever, parent_docs, ep, sem, proc, scope,
                use_ep, use_sem, use_proc
            )
            correct += result != "insufficient evidence"
            total_tokens += tokens

        rows.append(AblationRow(
            name=name,
            recall=recall,
            precision=precision,
            answered=correct / len(K.QUESTIONS),
            tokens=total_tokens,
        ))

    return rows


print(K.ablate(run_ablation(embedder)))

configuration                 recall  precision  answered    tokens
-------------------------------------------------------------------
no memory at all                  0%         0%        0%         0
fixed + keyword                  67%        33%       50%       480
recursive + keyword              83%        47%       67%       591
recursive + embeddings          100%        64%       83%       492
parent-child + embeddings       100%        64%       83%      1221
+ episodic                      100%        64%      100%      1469
+ episodic + semantic           100%        64%      100%      1493
everything                      100%        64%      100%      1699

Expect at least one layer to make things worse. That is a finding,
not a bug -- report it.


In [ ]:
# @title ✅ Solution — Task 4  { display-mode: "form" }

configuration                 recall  precision  answered    tokens
-------------------------------------------------------------------
no memory at all                  0%         0%        0%         0
fixed + keyword                  67%        33%       67%        88
recursive + keyword              83%        47%       83%       106
recursive + embeddings          100%        64%      100%        91
parent-child + embeddings       100%        64%      100%       240
+ episodic                      100%        64%      100%       269
+ episodic + semantic           100%        64%      100%       278
everything                      100%        64%      100%       312

Expect at least one layer to make things worse. That is a finding,
not a bug -- report it.


In [16]:
memo = """# Lab 5 — Memory Decision Memo

## 1. Which layer helped most, and on which questions?
Recursive embedding search raised the answered rate from 67% to 83%.
Episodic memory raised it to 100% by making Q6 answerable.

## 2. Which layer made things worse, and how do you know?
Parent-child retrieval increased estimated tokens from 492 to 1,221
without improving the answered rate. Semantic and procedural memory
added more tokens without improving it further.

## 3. Where did embeddings beat keywords?
For Q5, keyword search missed POL-BILLING because the question said
"charged twice" while the policy said "duplicate charges". Embedding
search found the correct policy. I would test both on more real questions
before choosing a production retrieval strategy.

## 4. What did episodic memory make possible?
It verified that cust-4417 had contacted support twice. A general policy
document cannot establish an individual customer's contact history.

## 5. Where does memory access sit in the authorisation model?
Every read and write requires a user scope enforced by the store.
The application must obtain the user identity from an authenticated
session, rather than trusting a user_id supplied in a request.

## 6. What did the measurement not tell you?
Five short documents and six exercise questions form only a smoke test.
The deterministic answer check does not assess generated answer quality.
Token counts are estimates, and only one embedding model was tested.
Changing that model would require re-indexing the vector collection.
"""

with open("decision_memo.md", "w", encoding="utf-8") as f:
    f.write(memo)

print("Created decision_memo.md")

Created decision_memo.md


In [17]:
import ast
from pathlib import Path

wanted = {
    "make_search_tool",
    "seed_memory",
    "isolation_test",
    "answer",
    "run_ablation",
}

found = {}

for cell_source in get_ipython().history_manager.input_hist_raw:
    try:
        tree = ast.parse(cell_source)
    except SyntaxError:
        continue

    for node in tree.body:
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
            if node.name in wanted:
                found[node.name] = ast.get_source_segment(cell_source, node)

missing = wanted - found.keys()
if missing:
    raise RuntimeError(f"Functions not found in executed cells: {sorted(missing)}")

header = """import json
import lab6_kit as K
from lab6_kit import (
    MemoryScope, EpisodicStore, SemanticStore,
    ProceduralStore, ScopeError, AblationRow
)

"""

source = header + "\n\n\n".join(found[name] for name in [
    "make_search_tool",
    "seed_memory",
    "isolation_test",
    "answer",
    "run_ablation",
]) + """

if __name__ == "__main__":
    ep, sem, proc = seed_memory()
    scope = MemoryScope(user_id="cust-4417")
    isolation_test(ep, sem)
    embedder = K.OllamaEmbedder()
    print(K.ablate(run_ablation(embedder)))
"""

Path("layla_memory.py").write_text(source, encoding="utf-8")
print("Created layla_memory.py with:", ", ".join(sorted(found)))

Created layla_memory.py with: answer, isolation_test, make_search_tool, run_ablation, seed_memory


In [18]:
!python layla_memory.py

configuration                 recall  precision  answered    tokens
-------------------------------------------------------------------
no memory at all                  0%         0%        0%         0
fixed + keyword                  67%        33%       50%       480
recursive + keyword              83%        47%       67%       591
recursive + embeddings          100%        64%       83%       492
parent-child + embeddings       100%        64%       83%      1221
+ episodic                      100%        64%      100%      1469
+ episodic + semantic           100%        64%      100%      1493
everything                      100%        64%      100%      1699

Expect at least one layer to make things worse. That is a finding,
not a bug -- report it.


### Read your table

1. **Which layer helped most?** It is probably not one of the memory layers.
2. **Which made things worse?** Look at the token column beside the answered
   column. A layer that adds context on every question and helps on none is a
   measurable regression — report it as a finding, not a failure.
3. **Where did embeddings beat keywords?** On how many of the six questions?
4. **Q6** asks whether this is the customer's second complaint. Which layer
   makes that answerable, and why can no document ever answer it?


## Stretch 1 — Poison your own memory

Write a plausible but false fact through the normal write path, then see how
long it survives and whether anything notices.

In [ ]:
sem.write(scope, "late_threshold_days", "1", asserted="2026-07-28")
print("retrieved as legitimate:", sem.get(scope, "late_threshold_days").value)

# Nothing distinguishes that from something the agent learned properly.
# A prompt injection lasts one turn; a poisoned memory PERSISTS -- it is
# retrieved again on future runs, for future users, looking exactly like
# knowledge. That is Week 10.

retrieved as legitimate: 1


## Stretch 2 — Put memory behind a Flask API

Your project needs an HTTP interface. Memory is a good place to practise,
because the scope key becomes a **request-level authorisation decision**
rather than a function argument.

```python
from flask import Flask, g, jsonify, request

app = Flask(__name__)

@app.post("/remember")
def remember():
    body = request.get_json()
    scope = MemoryScope(user_id=body["user_id"])      # from the REQUEST
    e = ep.write(scope, what=body["what"], action=body["action"],
                 outcome=body["outcome"], when=body["when"])
    return jsonify(episode_id=e.episode_id)

@app.get("/recall")
def recall():
    user = request.args.get("user_id")
    if not user:
        return jsonify(error="user_id required"), 400   # scope is mandatory
    scope = MemoryScope(user_id=user)
    return jsonify(episodes=[e.what for e in ep.recall(scope, k=5)])
```

Then ask the question that matters: **where does `user_id` come from in
production?** If it comes from the request body, any caller can read any
customer's memory. It has to come from an authenticated session — and that
is why the lecture called memory access authorisation rather than filtering.


## Submit

- this notebook, executed
- `layla_memory.py` — your tool, stores and ablation harness
- your ablation table
- `decision_memo.md` — the six questions below

### The decision memo

1. **Which layer helped most**, and on which questions?
2. **Which layer made things worse**, and how do you know?
3. **Where did embeddings beat keywords**, and what would you do in
   production?
4. **What did episodic memory make possible** that retrieval could not?
5. **Where does memory access sit in your authorisation model?**
6. **What did the measurement not tell you?**

Questions 2 and 6 carry the most marks. For question 6 be specific: five
short documents is not a corpus, six questions written by one person is a
smoke test, the reader is deterministic, and `nomic-embed-text` is one
embedding model — swapping it changes every number and invalidates the index.

### Before Week 7

Bring your ablation table, the layer that made things worse, and your
**project proposal draft — due Week 7**.